In [0]:
from pyspark.sql.types import *
data=[(1,'2024-01-01',"I1",10,1000),(2,"2024-01-15","I2",20,2000),(3,"2024-02-01","I3",10,1500),(4,"2024-02-15","I4",20,2500),(5,"2024-03-01","I5",30,3000),(6,"2024-03-10","I6",40,3500),(7,"2024-03-20","I7",20,2500),(8,"2024-03-30","I8",10,1000)]
schema=["SOId","SODate","ItemId","ItemQty","ItemValue"]
df1=spark.createDataFrame(data,schema)
display(df1)

SOId,SODate,ItemId,ItemQty,ItemValue
1,2024-01-01,I1,10,1000
2,2024-01-15,I2,20,2000
3,2024-02-01,I3,10,1500
4,2024-02-15,I4,20,2500
5,2024-03-01,I5,30,3000
6,2024-03-10,I6,40,3500
7,2024-03-20,I7,20,2500
8,2024-03-30,I8,10,1000


In [0]:
from pyspark.sql.functions import *

In [0]:
df2 = df1.withColumn("SODate", to_date(col("SODate"),"yyyy-MM-dd"))
display(df2)


SOId,SODate,ItemId,ItemQty,ItemValue
1,2024-01-01,I1,10,1000
2,2024-01-15,I2,20,2000
3,2024-02-01,I3,10,1500
4,2024-02-15,I4,20,2500
5,2024-03-01,I5,30,3000
6,2024-03-10,I6,40,3500
7,2024-03-20,I7,20,2500
8,2024-03-30,I8,10,1000


In [0]:
df3 = df2.withColumn("year_month", date_format(col("SODate"),"MMM-yyyy"))
display(df3)

SOId,SODate,ItemId,ItemQty,ItemValue,year_month
1,2024-01-01,I1,10,1000,Jan-2024
2,2024-01-15,I2,20,2000,Jan-2024
3,2024-02-01,I3,10,1500,Feb-2024
4,2024-02-15,I4,20,2500,Feb-2024
5,2024-03-01,I5,30,3000,Mar-2024
6,2024-03-10,I6,40,3500,Mar-2024
7,2024-03-20,I7,20,2500,Mar-2024
8,2024-03-30,I8,10,1000,Mar-2024


In [0]:
df4 = df3.withColumns({
    "month": month(col("SODate")),
    "year" : year(col("SODate"))
})

df4.display()

SOId,SODate,ItemId,ItemQty,ItemValue,year_month,month,year
1,2024-01-01,I1,10,1000,Jan-2024,1,2024
2,2024-01-15,I2,20,2000,Jan-2024,1,2024
3,2024-02-01,I3,10,1500,Feb-2024,2,2024
4,2024-02-15,I4,20,2500,Feb-2024,2,2024
5,2024-03-01,I5,30,3000,Mar-2024,3,2024
6,2024-03-10,I6,40,3500,Mar-2024,3,2024
7,2024-03-20,I7,20,2500,Mar-2024,3,2024
8,2024-03-30,I8,10,1000,Mar-2024,3,2024


In [0]:
df5 = df4.groupBy("Month","Year").agg(sum(col("ItemValue")).alias("sum"))
df5.display()

Month,Year,sum
1,2024,3000
2,2024,4000
3,2024,10000


In [0]:
df6 = df5.withColumn("prev_month_sum",lag("sum").over(Window.orderBy(col("Month"),col("Year"))))
df6.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Month,Year,sum,prev_month_sum
1,2024,3000,null
2,2024,4000,3000
3,2024,10000,4000


In [0]:
df7 = df6.withColumn("per_change",(col("sum")-col("prev_month_sum"))/col("sum")*100)
df7.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Month,Year,sum,prev_month_sum,per_change
1,2024,3000,null,null
2,2024,4000,3000,25.0
3,2024,10000,4000,60.0
